# 05 - Spider Walking DMD

A short, self-contained example on a different animal: a walking spider.
It follows the same workflow as `02_bird_flight_dmd.ipynb`, so the steps will look familiar.

The spider skeleton is built into `morphing_birds`, so a single packaged motion file is all you need.
The highlight is the last step: animating a single DMD mode pair to see the walking gait on its own.


## Setup

Run this once at the start of Colab. If you are working locally in the repository environment, you can skip the install cell and run the imports.


In [ ]:
# Run this once at the start of Colab.
!pip install -q uv
!uv pip install --system -q git+https://github.com/LydiaFrance/dmd-workshop.git@main

from animal_dmd import setup_workshop
setup_workshop();

In [ ]:
import warnings

import numpy as np
from birddmd import reconstruct, run_dmd
from IPython.display import display

import morphing_birds as mb
from morphing_birds import animate_plotly, animate_plotly_compare

from animal_dmd import (
    load_spider_motion,
    plot_dmd_pair_traces,
    plot_marker_traces,
    plot_reconstruction_marker_traces,
    plot_unit_circle_eigenvalues,
    print_dmd_summary,
)

np.set_printoptions(precision=3, suppress=True)
warnings.filterwarnings(
    "ignore",
    message="Input data condition number.*",
    category=UserWarning,
)

## Load the Spider Walking Data

One packaged window of a walking spider. `load_spider_motion` returns the `Animal3D` skeleton for animation plus the motion array, times, marker names, and the frame interval `dt`. Times are in frames, so `dt = 1` and the DMD frequencies below are in cycles per frame.


In [ ]:
spider, markers, times, marker_names, dt = load_spider_motion()

print("Loaded the spider walking dataset")
print(f"{markers.shape[0]} frames, {markers.shape[1]} markers, {markers.shape[2]} coordinates")
print(f"{dt:.4f} frames between samples")

## Choose a Trace to Watch

DMD fits the whole motion, but a single marker coordinate is the simplest reconstruction check. A leg-tip claw moving up and down is a clear walking signal.


In [ ]:
selected_marker = "claw1"
selected_dimension = 2  # 0=x, 1=y, 2=z
dimension_labels = ["x", "y", "z"]

selected_marker_index = marker_names.index(selected_marker)
trace_label = f"{selected_marker} {dimension_labels[selected_dimension]}"

plot_marker_traces(
    times,
    {trace_label: markers[:, selected_marker_index, selected_dimension]},
    title=f"Input trace: {trace_label}",
)

## Animate the Raw Motion

The spider skeleton lets us watch the recorded walking motion before any analysis.


In [ ]:
display(animate_plotly(spider, keypoints_frames=markers, axes_visible=True))

## Run DMD

Same settings as the spider analysis notebook: a small number of modes with conjugate pairs, so the oscillatory walking motion comes out as clean mode pairs.


In [ ]:
mean_pose = markers.mean(axis=0, keepdims=True)

dmd_rank = 6
dmd_result = run_dmd(
    data=markers,
    times=times,
    n_modes=dmd_rank,
    d=2,
    eig_constraints={"conjugate_pairs"},
    average_shape=mean_pose,
    n_markers=markers.shape[1],
    verbose=False,
)

dmd_reconstruction = dmd_result.reconstruction
reconstruction_times = times[1:]
reconstruction_rmse = np.sqrt(np.mean((markers[1:] - dmd_reconstruction) ** 2))

growth_per_second = dmd_result.eigenvalues.real
frequency_hz = dmd_result.eigenvalues.imag / (2 * np.pi)

print_dmd_summary(
    growth_per_second,
    frequency_hz,
    reconstruction_rmse,
    label=f"rank-{dmd_rank} reconstruction",
)

## Check the Reconstruction

Before interpreting modes, check whether the full DMD reconstruction matches the data it was fitted to. The reconstruction starts at `times[1]` because DMD learns how one frame advances to the next.


In [ ]:
plot_reconstruction_marker_traces(
    reconstruction_times,
    {trace_label: markers[1:, selected_marker_index, selected_dimension]},
    {trace_label: dmd_reconstruction[:, selected_marker_index, selected_dimension]},
    title=f"DMD reconstruction check: {trace_label}",
)

In [ ]:
per_frame_eigenvalues = np.exp(dmd_result.eigenvalues * dt)
plot_unit_circle_eigenvalues(per_frame_eigenvalues)

## Inspect DMD Mode Pairs

The walking gait appears as conjugate pairs. List the pair frequencies first, then look at how each pair reconstructs the chosen trace.


In [ ]:
pair_frequencies_hz = np.array(
    [dmd_result.pair_frequency(pair_index) for pair_index in range(dmd_result.n_pairs)]
)

for pair_index, pair_frequency in enumerate(pair_frequencies_hz):
    print(f"pair {pair_index}: {pair_frequency:.4f} cycles/frame")

In [ ]:
selected_pairs = list(range(dmd_result.n_pairs))

pair_reconstructions = {
    pair_index: reconstruct(dmd_result, pairs=[pair_index])
    for pair_index in selected_pairs
}
pair_traces = {
    pair_index: pair_motion[:, selected_marker_index, selected_dimension]
    for pair_index, pair_motion in pair_reconstructions.items()
}

plot_dmd_pair_traces(
    reconstruction_times,
    markers[1:, selected_marker_index, selected_dimension],
    pair_traces,
    pair_frequencies_hz=pair_frequencies_hz,
    trace_label=trace_label,
)

## Animate the Walking Mode

Finally, animate a single mode pair on the spider skeleton. Each pair is one oscillatory pattern, so this shows that part of the walking motion on its own. Change `pair_index` to inspect a different pair.


In [ ]:
mode_colours = ["blue", "green", "orange", "purple", "cyan"]
pair_index = 0

display(
    animate_plotly_compare(
        spider,
        keypoints_frames_list=[pair_reconstructions[pair_index]],
        colours=[mode_colours[pair_index % len(mode_colours)]],
    )
)

## What to Adjust Next

- Change `pair_index` in the animation to watch a different mode pair.
- Increase `dmd_rank` if the reconstruction misses obvious motion; decrease it if the modes look noisy.
- Switch `selected_marker` and `selected_dimension` to follow a different leg.
- Compare the pair frequencies with the spider's step rate to identify the gait mode.
